# Hinglish Smart Writing Assistant
Roman Hinglish -> native script -> improved phrasing -> English

**Pipeline:** `LLM transliteration (+ guardrails + lexicon)` -> `LLM phrasing improvement (guarded)` -> `IndicTrans2 translation`


## 1. Install

In [1]:
!pip -q install "transformers==4.46.3" "accelerate>=0.34" sentencepiece gradio jiwer sacrebleu pandas IndicTransToolkit bitsandbytes bert-score datasets

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


## 2. Config and imports

In [2]:
import re, time, json, random, unicodedata
from collections import Counter
import torch
import pandas as pd
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM,
                          BitsAndBytesConfig)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# LLM for transliteration + phrasing.
# GPU: Qwen2.5-7B in 4-bit (fits a T4). CPU: small model (slow, for testing only).
# To try BharatGen Param1: set LLM_NAME to its repo id from huggingface.co/bharatgenai,
# accept the licence and run: from huggingface_hub import login; login()
LLM_NAME = "Qwen/Qwen2.5-7B-Instruct" if DEVICE == "cuda" else "Qwen/Qwen2.5-1.5B-Instruct"
USE_4BIT = DEVICE == "cuda"

# IndicTrans2 distilled 200M (Indic -> English). Use "ai4bharat/indictrans2-indic-en-1B" for higher quality.
TRANS_NAME = "ai4bharat/indictrans2-indic-en-dist-200M"

# language name -> (IndicXlit code, IndicTrans2/FLORES code)
LANGS = {
    "Hindi":    ("hi", "hin_Deva"),
    "Marathi":  ("mr", "mar_Deva"),
    "Bengali":  ("bn", "ben_Beng"),
    "Tamil":    ("ta", "tam_Taml"),
    "Telugu":   ("te", "tel_Telu"),
    "Kannada":  ("kn", "kan_Knda"),
    "Gujarati": ("gu", "guj_Gujr"),
}

Device: cuda


## 3. Load models

In [3]:
# --- LLM ---
llm_tok = AutoTokenizer.from_pretrained(LLM_NAME)
try:
    if USE_4BIT:
        llm = AutoModelForCausalLM.from_pretrained(
            LLM_NAME, device_map="auto",
            quantization_config=BitsAndBytesConfig(load_in_4bit=True,
                                                   bnb_4bit_compute_dtype=torch.float16))
    else:
        llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=torch.float32).to(DEVICE)
except Exception as e:
    print("Primary LLM load failed, falling back to Qwen2.5-3B fp16:", e)
    LLM_NAME = "Qwen/Qwen2.5-3B-Instruct"
    llm_tok = AutoTokenizer.from_pretrained(LLM_NAME)
    llm = AutoModelForCausalLM.from_pretrained(
        LLM_NAME, torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32).to(DEVICE)
llm.eval()
print("LLM:", LLM_NAME)

# --- IndicTrans2 ---
tr_tok = AutoTokenizer.from_pretrained(TRANS_NAME, trust_remote_code=True)
tr_model = AutoModelForSeq2SeqLM.from_pretrained(
    TRANS_NAME, trust_remote_code=True,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32).to(DEVICE)
tr_model.eval()

try:
    try:
        from IndicTransToolkit.processor import IndicProcessor
    except ImportError:
        from IndicTransToolkit import IndicProcessor
    ip = IndicProcessor(inference=True)
    print("IndicTransToolkit loaded.")
except Exception as e:
    ip = None
    print("IndicTransToolkit unavailable, using manual tagging fallback:", e)
print("Models ready.")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

... (18 outputs left out)

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-en-dist-200M:
- modeling_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/913M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

IndicTransToolkit loaded.
Models ready.


## 4. Core functions


In [9]:
# ---------------- LLM helper ----------------
def llm_chat(messages, max_new_tokens=512):
    prompt = llm_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    enc = llm_tok(prompt, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        out = llm.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                           pad_token_id=llm_tok.eos_token_id)
    return llm_tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def clean(s):
    s = s.strip()
    s = s.strip('"\u201c\u201d`').strip()
    return s

PUNCT = ".,!?;:\"'()"
def split_punct(w):
    core = w.strip(PUNCT)
    if not core:
        return w, "", ""
    i = w.find(core)
    return w[:i], core, w[i + len(core):]

def nums(s):
    return re.findall(r"\d+", s)

# ---------------- Few-shot examples (Hindi) ----------------
# (roman, native with loanwords in Devanagari, native with English words kept in Latin)
# IMPORTANT: keep these different from your TEST sentences.
HI_SHOTS_DATA = [
    ("mujhe subah 7 baje station pahunchna hai", "मुझे सुबह 7 बजे स्टेशन पहुँचना है", "मुझे सुबह 7 बजे station पहुँचना है"),
    ("woh kal bazaar se sabzi layega", "वो कल बाज़ार से सब्ज़ी लाएगा", "वो कल बाज़ार से सब्ज़ी लाएगा"),
    ("tumne kitab kahan rakhi hai?", "तुमने किताब कहाँ रखी है?", "तुमने किताब कहाँ रखी है?"),
    ("hamare office mein aaj party hai", "हमारे ऑफिस में आज पार्टी है", "हमारे office में आज party है"),
    ("please mera laptop theek kar do", "प्लीज़ मेरा लैपटॉप ठीक कर दो", "please मेरा laptop ठीक कर दो"),
    ("usne mujhe kal raat call kiya tha", "उसने मुझे कल रात कॉल किया था", "उसने मुझे कल रात call किया था"),
    ("main doodh nahi peeta, juice pasand hai", "मैं दूध नहीं पीता, जूस पसंद है", "मैं दूध नहीं पीता, juice पसंद है"),
    ("kya tum train ka ticket book kar chuke ho?", "क्या तुम ट्रेन का टिकट बुक कर चुके हो?", "क्या तुम train का ticket book कर चुके हो?"),
    ("mujhe 3 din ki chhutti chahiye", "मुझे 3 दिन की छुट्टी चाहिए", "मुझे 3 दिन की छुट्टी चाहिए"),
    ("yaar pichhle hafte ka match bahut zabardast tha", "यार पिछले हफ्ते का मैच बहुत ज़बरदस्त था", "यार पिछले हफ्ते का match बहुत ज़बरदस्त था"),
]

# ---------------- Lexicon of high-frequency Hindi words ----------------
# General function words / very common words only. Add domain words only from your DEV set,
# never from your TEST set (otherwise your score is inflated).
LEXICON = {
    "hai": "है", "hain": "हैं", "ho": "हो", "hoon": "हूँ", "tha": "था", "thi": "थी", "the": "थे",
    "nahi": "नहीं", "nahin": "नहीं", "mat": "मत", "ka": "का", "ki": "की", "ke": "के", "ko": "को",
    "se": "से", "mein": "में", "me": "में", "par": "पर", "aur": "और", "ya": "या", "lekin": "लेकिन",
    "kyunki": "क्योंकि", "to": "तो", "bhi": "भी", "hi": "ही", "main": "मैं", "mai": "मैं",
    "mujhe": "मुझे", "mera": "मेरा", "meri": "मेरी", "mere": "मेरे", "tum": "तुम", "tumhe": "तुम्हें",
    "tumhara": "तुम्हारा", "aap": "आप", "aapka": "आपका", "hum": "हम", "hamara": "हमारा",
    "woh": "वो", "vo": "वो", "yeh": "यह", "ye": "ये", "kya": "क्या", "kyun": "क्यों", "kyu": "क्यों",
    "kaise": "कैसे", "kahan": "कहाँ", "kab": "कब", "kitna": "कितना", "kitne": "कितने", "kaun": "कौन",
    "kal": "कल", "aaj": "आज", "abhi": "अभी", "bahut": "बहुत", "thoda": "थोड़ा", "accha": "अच्छा",
    "theek": "ठीक", "sab": "सब", "saath": "साथ", "baje": "बजे", "subah": "सुबह", "raat": "रात",
    "kar": "कर", "karna": "करना", "karo": "करो", "jaana": "जाना", "ja": "जा", "raha": "रहा",
    "rahi": "रही", "rahe": "रहे", "gaya": "गया", "gayi": "गई", "chahiye": "चाहिए", "dobara": "दोबारा",
}

# ---------------- Step 1: transliteration ----------------
def transliterate_llm_raw(text, lang, keep_english=False, strict=False):

    rule = (
        "Keep English words in Latin script exactly as written."
        if keep_english else
        "Write English loanwords in the native script as they are pronounced."
    )

    system = (
        f"You convert Romanized {lang} text into {lang} native script. "
        "Transliterate based on pronunciation and sentence context. "
        "Do not translate, summarize, explain, or paraphrase. "
        "Preserve the original meaning, word order, numbers, and punctuation. "
        "Do not add or remove words. "
        f"{rule} "
        "Return only the transliterated text."
    )

    if strict:
        system += (
            " Be especially careful with every input word. "
            "Do not skip, combine, or repeat words. "
            "Preserve the original sentence structure."
        )

    shots = []

    if lang == "Hindi":
        for s, loan, keep in HI_SHOTS_DATA:
            shots += [
                {"role": "user", "content": s},
                {
                    "role": "assistant",
                    "content": keep if keep_english else loan
                }
            ]

    outs = []

    for line in text.split("\n"):

        if not line.strip():
            outs.append("")
            continue

        msgs = (
            [{"role": "system", "content": system}]
            + shots
            + [{"role": "user", "content": line}]
        )

        outs.append(clean(llm_chat(msgs)))

    return "\n".join(outs)

_word_cache = {}
def xlit_word(w, lang):
    pre, core, post = split_punct(w)
    if not core or core.isdigit():
        return w
    k = (lang, core.lower())
    if k not in _word_cache:
        if lang == "Hindi" and core.lower() in LEXICON:
            _word_cache[k] = LEXICON[core.lower()]
        else:
            _word_cache[k] = clean(llm_chat([
                {"role": "system", "content": f"Transliterate this single Romanized {lang} word into {lang} "
                                              f"native script by sound. Do not translate. Output only the word."},
                {"role": "user", "content": core}], max_new_tokens=16))
    return pre + _word_cache[k] + post

def transliterate_llm(text, lang, keep_english=False, robust=True):

    if not robust:
        return transliterate_llm_raw(
            text, lang, keep_english
        )

    outs = []

    for line in text.split("\n"):

        if not line.strip():
            outs.append("")
            continue

        src = line.split()

        # First transliteration attempt
        first = transliterate_llm_raw(
            line, lang, keep_english
        )

        cand = first.split()

        # Check whether token alignment is reliable
        valid = (
            len(cand) == len(src)
            and nums(line) == nums(first)
        )

        # Retry the complete sentence if alignment fails
        if not valid:

            retry = transliterate_llm_raw(
                line, lang, keep_english, strict=True
            )

            retry_tokens = retry.split()

            retry_valid = (
                len(retry_tokens) == len(src)
                and nums(line) == nums(retry)
                and bool(retry.strip())
            )

            if retry_valid:
                cand = retry_tokens
            else:
                # Avoid potentially incorrect word-level fallback
                outs.append(first)
                continue

        # Apply lexicon only when English preservation is disabled
        if lang == "Hindi" and not keep_english:

            for i, w in enumerate(src):

                pre, core, post = split_punct(w)

                if core.lower() in LEXICON:
                    cand[i] = (
                        pre + LEXICON[core.lower()] + post
                    )

        outs.append(" ".join(cand))

    return "\n".join(outs)

_xlit_engines = {}
def transliterate_indicxlit(text, lang):
    try:
        from ai4bharat.transliteration import XlitEngine
    except ImportError:
        raise RuntimeError("IndicXlit is not installed. Run the optional IndicXlit cell, or use the LLM engine.")
    code = LANGS[lang][0]
    if code not in _xlit_engines:
        _xlit_engines[code] = XlitEngine(code, beam_width=10, src_script_type="en")
    outs = []
    for line in text.split("\n"):
        if not line.strip():
            outs.append("")
            continue
        r = _xlit_engines[code].translit_sentence(line)
        outs.append(r[code] if isinstance(r, dict) else r)
    return "\n".join(outs)

# ---------------- Step 2: improve phrasing (guarded) ----------------
def improve_phrasing(text, lang, tone="neutral"):

    system = (
        f"You are a careful {lang} writing assistant. "
        f"Improve the user's text in a {tone} tone. "
        "Correct grammar and spelling while keeping the original meaning. "
        "Do not change negation, tense, gender, person, or intent. "
        "Preserve all names, numbers, dates, and important details. "
        "Do not add or remove information. "
        "Do not translate into another language. "
        "Return only the improved text."
    )

    negative_words = {"नहीं", "नही", "मत", "ना"}

    def has_negation(s):
        words = re.findall(r"\S+", s)
        return any(w.strip(".,!?;") in negative_words for w in words)

    outs = []

    for line in text.split("\n"):

        if not line.strip():
            outs.append("")
            continue

        new = clean(
            llm_chat([
                {"role": "system", "content": system},
                {"role": "user", "content": line}
            ])
        )

        ratio = len(new) / max(len(line), 1)

        reject = (
            not new
            or nums(new) != nums(line)
            or has_negation(new) != has_negation(line)
            or ratio < 0.5
            or ratio > 2.0
        )

        outs.append(line if reject else new)

    return "\n".join(outs)

# ---------------- Step 3: translate to English ----------------
def _translate_batch(sents, src, tgt="eng_Latn"):
    if ip is not None:
        batch = ip.preprocess_batch(sents, src_lang=src, tgt_lang=tgt)
    else:
        batch = [f"{src} {tgt} {s}" for s in sents]
    inputs = tr_tok(batch, truncation=True, padding="longest",
                    return_tensors="pt", return_attention_mask=True).to(DEVICE)
    with torch.no_grad():
        gen = tr_model.generate(**inputs, use_cache=True, min_length=0, max_length=256,
                                num_beams=5, num_return_sequences=1)
    with tr_tok.as_target_tokenizer():
        dec = tr_tok.batch_decode(gen.detach().cpu().tolist(), skip_special_tokens=True,
                                  clean_up_tokenization_spaces=True)
    return ip.postprocess_batch(dec, lang=tgt) if ip is not None else dec

def translate_to_english(text, lang):
    lines = text.split("\n")
    idx = [i for i, l in enumerate(lines) if l.strip()]
    if not idx:
        return text
    src = LANGS[lang][1]
    outs = []
    for k in range(0, len(idx), 8):
        outs += _translate_batch([lines[i] for i in idx[k:k + 8]], src)
    res = list(lines)
    for i, o in zip(idx, outs):
        res[i] = o
    return "\n".join(res)

# ---------------- Full pipeline ----------------
def available_engines():
    engines = ["LLM"]
    try:
        import ai4bharat.transliteration  # noqa
        engines.append("IndicXlit")
    except Exception:
        pass
    return engines

def pipeline(text, lang="Hindi", engine="LLM", improve=True, tone="neutral",
             to_english=True, keep_english=False, robust=True):
    text = (text or "").strip()
    if not text:
        return "", "", "", "Please enter some text."
    t0 = time.time()
    try:
        native = (transliterate_indicxlit(text, lang) if engine == "IndicXlit"
                  else transliterate_llm(text, lang, keep_english, robust))
        improved = improve_phrasing(native, lang, tone) if improve else ""
        base = improved if improve else native
        english = translate_to_english(base, lang) if to_english else ""
        return native, improved, english, f"Done in {time.time() - t0:.1f}s"
    except Exception as e:
        return "", "", "", f"Error: {e}"

# quick smoke test
print(pipeline("mujhe kal office jaana hai par mera phone kharab ho gaya hai"))

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


('मुझे कल ऑफिस जाना है पर मेरा फोन क्हारब हो गया है', 'मुझे कल ऑफिस जाना है, लेकिन मेरा फोन क्हारब हो गया है।', 'I have to go to the office tomorrow, but my phone is dead.', 'Done in 10.8s')


/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py:4114: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


## 5. Test data (50 cases) and DEV / TEST split
The references were written by hand. Use **DEV** to tune the lexicon, few-shot examples and prompts; report final numbers on **TEST** only.


In [10]:
DATA = [
    # (hinglish, ref_native, ref_english)
    ("mujhe kal subah chhe baje uthna hai", "मुझे कल सुबह छह बजे उठना है", "I have to wake up at six tomorrow morning."),
    ("aap kaise hain", "आप कैसे हैं", "How are you?"),
    ("main aaj office nahi jaa raha hoon", "मैं आज ऑफिस नहीं जा रहा हूँ", "I am not going to the office today."),
    ("kya tum mere saath chai peena chahoge", "क्या तुम मेरे साथ चाय पीना चाहोगे", "Would you like to have tea with me?"),
    ("mera phone kharab ho gaya hai", "मेरा फोन खराब हो गया है", "My phone has stopped working."),
    ("kal exam hai, bahut tension ho rahi hai", "कल एग्ज़ाम है, बहुत टेंशन हो रही है", "There is an exam tomorrow, I am very stressed."),
    ("yeh assignment kab tak submit karna hai", "यह असाइनमेंट कब तक सबमिट करना है", "By when do we have to submit this assignment?"),
    ("mausam aaj bahut accha hai", "मौसम आज बहुत अच्छा है", "The weather is very nice today."),
    ("tum kaise ho", "तुम कैसे हो", "How are you?"),
    ("main ghar ja raha hoon", "मैं घर जा रहा हूँ", "I am going home."),
    ("kya aap meri madad kar sakte hain", "क्या आप मेरी मदद कर सकते हैं", "Can you help me?"),
    ("please mujhe report kal tak bhej do", "प्लीज़ मुझे रिपोर्ट कल तक भेज दो", "Please send me the report by tomorrow."),
    ("hum sab saath mein lunch karenge", "हम सब साथ में लंच करेंगे", "We will all have lunch together."),
    ("bus station kahan hai", "बस स्टेशन कहाँ है", "Where is the bus station?"),
    ("bhai kal ki class kitne baje hai", "भाई कल की क्लास कितने बजे है", "Bro, what time is tomorrow's class?"),
    ("yaar tu itna late kyun aaya", "यार तू इतना लेट क्यों आया", "Dude, why did you come so late?"),
    ("bahut mast gaana hai yaar", "बहुत मस्त गाना है यार", "This is a really cool song, dude."),
    ("aaj raat ko movie dekhne chalte hain", "आज रात को मूवी देखने चलते हैं", "Let's go watch a movie tonight."),
    ("pareshaan mat ho, sab theek ho jayega", "परेशान मत हो, सब ठीक हो जाएगा", "Don't worry, everything will be fine."),
    ("chalo phir milte hain, bye", "चलो फिर मिलते हैं, बाय", "Okay, let's meet again, bye."),
    ("kya tumne khana khaya", "क्या तुमने खाना खाया", "Did you eat?"),
    ("mujhe bhook lagi hai", "मुझे भूख लगी है", "I am hungry."),
    ("tumhara naam kya hai", "तुम्हारा नाम क्या है", "What is your name?"),
    ("mujhe samajh nahi aaya, dobara samjhao", "मुझे समझ नहीं आया, दोबारा समझाओ", "I did not understand, explain again."),
    ("WiFi kaam nahi kar raha hai", "वाईफाई काम नहीं कर रहा है", "The WiFi is not working."),
    ("meri battery khatam hone wali hai", "मेरी बैटरी खत्म होने वाली है", "My battery is about to run out."),
    ("mujhe is project mein aapki help chahiye", "मुझे इस प्रोजेक्ट में आपकी हेल्प चाहिए", "I need your help with this project."),
    ("kal se main gym join karunga", "कल से मैं जिम जॉइन करूँगा", "I will join the gym from tomorrow."),
    ("hamare college mein kal fest hai", "हमारे कॉलेज में कल फेस्ट है", "There is a fest in our college tomorrow."),
    ("teacher ne homework diya hai", "टीचर ने होमवर्क दिया है", "The teacher has given homework."),
    ("mujhe email bhejna hai manager ko", "मुझे ईमेल भेजना है मैनेजर को", "I need to send an email to the manager."),
    ("uska result aaj aane wala hai", "उसका रिजल्ट आज आने वाला है", "His result is going to come out today."),
    ("kripya apna form kal tak jama kijiye", "कृपया अपना फॉर्म कल तक जमा कीजिए", "Please submit your form by tomorrow."),
    ("sir, mujhe leave chahiye kyunki meri tabiyat kharab hai", "सर, मुझे लीव चाहिए क्योंकि मेरी तबीयत खराब है", "Sir, I need leave because I am not well."),
    ("meeting subah 10 baje shuru hogi", "मीटिंग सुबह 10 बजे शुरू होगी", "The meeting will start at 10 in the morning."),
    ("mujhe thoda time chahiye", "मुझे थोड़ा टाइम चाहिए", "I need a little time."),
    ("dhanyavaad, aapka din shubh ho", "धन्यवाद, आपका दिन शुभ हो", "Thank you, have a good day."),
    ("mere paas 500 rupaye hain", "मेरे पास 500 रुपये हैं", "I have 500 rupees."),
    ("mujhe 15 tarikh ko chhutti chahiye", "मुझे 15 तारीख को छुट्टी चाहिए", "I need leave on the 15th."),
    ("train platform number 3 par aayegi", "ट्रेन प्लेटफॉर्म नंबर 3 पर आएगी", "The train will arrive on platform number 3."),
    ("main kal Bangalore ja raha hoon", "मैं कल बैंगलोर जा रहा हूँ", "I am going to Bangalore tomorrow."),
    ("yeh kitne ka hai", "यह कितने का है", "How much is this?"),
    ("barish ki wajah se flight late ho gayi", "बारिश की वजह से फ्लाइट लेट हो गई", "The flight got delayed because of rain."),
    ("dukaan band ho chuki hai", "दुकान बंद हो चुकी है", "The shop has already closed."),
    ("mujhe sir dard ho raha hai, doctor ke paas jaana hai", "मुझे सिर दर्द हो रहा है, डॉक्टर के पास जाना है", "I have a headache, I need to go to the doctor."),
    ("mai tumhe kal subah call karunga", "मैं तुम्हें कल सुबह कॉल करूँगा", "I will call you tomorrow morning."),
    ("kyaa tum mere saath aaoge", "क्या तुम मेरे साथ आओगे", "Will you come with me?"),
    ("kal tak ka time do, main kar dunga", "कल तक का टाइम दो, मैं कर दूँगा", "Give me until tomorrow, I will do it."),
    ("agar kal barish nahi hui to hum sab picnic par jayenge aur wahan khana bhi banayenge",
     "अगर कल बारिश नहीं हुई तो हम सब पिकनिक पर जाएँगे और वहाँ खाना भी बनाएँगे",
     "If it does not rain tomorrow, we will all go on a picnic and cook food there too."),
    ("maine kal raat bahut der tak padhai ki lekin phir bhi paper mushkil laga",
     "मैंने कल रात बहुत देर तक पढ़ाई की लेकिन फिर भी पेपर मुश्किल लगा",
     "I studied until very late last night but the paper still felt difficult."),
        # Additional edge cases

    ("main nahi aa sakta",
     "मैं नहीं आ सकता",
     "I cannot come."),

    ("tum kal mere ghar aaoge?",
     "तुम कल मेरे घर आओगे?",
     "Will you come to my house tomorrow?"),

    ("mujhe 2 din ka samay chahiye",
     "मुझे 2 दिन का समय चाहिए",
     "I need two days."),

    ("kripya mujhe jawab dijiye.",
     "कृपया मुझे जवाब दीजिए।",
     "Please give me an answer."),

    ("aaj mausam bahut accha hai!",
     "आज मौसम बहुत अच्छा है!",
     "The weather is very nice today!"),

    ("main kal office nahi ja paunga",
     "मैं कल ऑफिस नहीं जा पाऊँगा",
     "I will not be able to go to the office tomorrow."),

    ("kya aap mujhe bata sakte hain?",
     "क्या आप मुझे बता सकते हैं?",
     "Can you tell me?"),

    ("mera kaam abhi tak pura nahi hua",
     "मेरा काम अभी तक पूरा नहीं हुआ",
     "My work is not completed yet."),

    ("woh ghar par hai aur main bahar hoon",
     "वो घर पर है और मैं बाहर हूँ",
     "He is at home and I am outside."),

    ("please mujhe ek minute do",
     "प्लीज़ मुझे एक मिनट दो",
     "Please give me a minute.")
]
# Or load your own CSV (columns: hinglish, ref_native, ref_en):
# d = pd.read_csv("my_tests.csv"); DATA = list(d[["hinglish","ref_native","ref_en"]].itertuples(index=False, name=None))

_idx = list(range(len(DATA)))
random.Random(42).shuffle(_idx)
n_dev = max(1, round(0.3 * len(DATA)))
DEV  = [DATA[i] for i in _idx[:n_dev]]
TEST = [DATA[i] for i in _idx[n_dev:]]
print(f"{len(DATA)} cases -> DEV {len(DEV)} / TEST {len(TEST)}")

60 cases -> DEV 18 / TEST 42


## 6. Evaluation (CER, word accuracy, BLEU, chrF, BERTScore, latency)

In [11]:
import jiwer, sacrebleu

def norm(s):
    s = unicodedata.normalize("NFD", s)
    s = s.replace("\u093c", "")                 # drop nukta
    s = unicodedata.normalize("NFC", s).replace("\u0901", "\u0902")   # chandrabindu -> anusvara
    s = re.sub(r"[\u0964.,!?;:\"'()]", "", s)
    return re.sub(r"\s+", " ", s).strip()

def bert_f1(preds, refs):
    try:
        from bert_score import score
        _, _, F = score(preds, refs, lang="en", model_type="distilbert-base-uncased", verbose=False)
        return F.mean().item() * 100
    except Exception as e:
        print("BERTScore skipped:", e)
        return float("nan")

def run_eval(data, label, engine="LLM", improve=False,
             robust=True, verbose=True):

    rows = []

    for src, ref_n, ref_e in data:

        t0 = time.time()

        native, improved, english, status = pipeline(
            src,
            "Hindi",
            engine,
            improve=improve,
            to_english=True,
            robust=robust
        )

        # Evaluate transliteration separately
        rn = norm(ref_n)
        pn = norm(native)

        # Correct WER-based word accuracy
        wer = (
            jiwer.wer(rn, pn)
            if rn else (0.0 if not pn else 1.0)
        )

        word_acc = max(0.0, 1.0 - wer)

        # Check whether phrasing changed the output
        improved_changed = (
            bool(improved)
            and norm(improved) != norm(native)
        )

        rows.append({
            "input": src,
            "ref_native": ref_n,
            "pred_native": native,
            "improved": improved,
            "ref_en": ref_e,
            "pred_en": english,

            "cer": (
                jiwer.cer(rn, pn)
                if rn else (0.0 if not pn else 1.0)
            ),

            "wer": wer,
            "word_acc": word_acc,
            "exact": rn == pn,

            "latin_left": bool(
                re.search(r"[A-Za-z]", native)
            ),

            "improved_changed": improved_changed,

            "latency_s": round(time.time() - t0, 2),
            "status": status
        })

    df = pd.DataFrame(rows)

    summ = {
        "config": label,
        "CER": df.cer.mean(),
        "WER": df.wer.mean(),
        "word_acc": df.word_acc.mean(),
        "exact": df.exact.mean(),

        "BLEU": sacrebleu.corpus_bleu(
            df.pred_en.tolist(),
            [df.ref_en.tolist()]
        ).score,

        "chrF": sacrebleu.corpus_chrf(
            df.pred_en.tolist(),
            [df.ref_en.tolist()]
        ).score,

        "BERTScore": bert_f1(
            df.pred_en.tolist(),
            df.ref_en.tolist()
        ),

        "latin_left": df.latin_left.mean(),
        "phrasing_changed": df.improved_changed.mean(),
        "latency_s": df.latency_s.mean()
    }

    if verbose:
        print({
            k: round(v, 3) if isinstance(v, float) else v
            for k, v in summ.items()
        })

    return df, summ

CONFIGS = [
    ("1. LLM raw",              dict(engine="LLM", improve=False, robust=False)),
    ("2. LLM + guard/lexicon",  dict(engine="LLM", improve=False, robust=True)),
    ("3. + phrasing improve",   dict(engine="LLM", improve=True,  robust=True)),
]
if "IndicXlit" in available_engines():
    CONFIGS.append(("4. IndicXlit", dict(engine="IndicXlit", improve=False, robust=True)))

# Report on TEST. (Run with data=DEV while you tune the lexicon / few-shots.)
EVAL_DATA = TEST
results, summaries = {}, []
for label, kw in CONFIGS:
    results[label], s = run_eval(EVAL_DATA, label, **kw)
    summaries.append(s)

summary_df = pd.DataFrame(summaries).round(3)
summary_df.to_csv("summary.csv", index=False)
for k, d in results.items():
    d.to_csv("results_" + re.sub(r"\W+", "_", k) + ".csv", index=False)
summary_df

{'config': '1. LLM raw', 'CER': np.float64(0.091), 'WER': np.float64(0.127), 'word_acc': np.float64(0.873), 'exact': np.float64(0.429), 'BLEU': 48.005, 'chrF': 69.011, 'BERTScore': 93.142, 'latin_left': np.float64(0.119), 'phrasing_changed': np.float64(0.0), 'latency_s': np.float64(2.52)}
{'config': '2. LLM + guard/lexicon', 'CER': np.float64(0.076), 'WER': np.float64(0.108), 'word_acc': np.float64(0.892), 'exact': np.float64(0.5), 'BLEU': 45.296, 'chrF': 67.628, 'BERTScore': 92.921, 'latin_left': np.float64(0.071), 'phrasing_changed': np.float64(0.0), 'latency_s': np.float64(2.469)}
{'config': '3. + phrasing improve', 'CER': np.float64(0.076), 'WER': np.float64(0.108), 'word_acc': np.float64(0.892), 'exact': np.float64(0.5), 'BLEU': 46.391, 'chrF': 67.278, 'BERTScore': 93.644, 'latin_left': np.float64(0.071), 'phrasing_changed': np.float64(0.69), 'latency_s': np.float64(4.206)}


,config,CER,WER,word_acc,exact,BLEU,chrF,BERTScore,latin_left,phrasing_changed,latency_s
0,1. LLM raw,0.091,0.127,0.873,0.429,48.005,69.011,93.142,0.119,0.00,2.520
1,2. LLM + guard/lexicon,0.076,0.108,0.892,0.500,45.296,67.628,92.921,0.071,0.00,2.469
2,3. + phrasing improve,0.076,0.108,0.892,0.500,46.391,67.278,93.644,0.071,0.69,4.206


## 7. Failure / error analysis

In [12]:
best = "2. LLM + guard/lexicon"
d = results[best]

print("Worst 10 by CER:")
display(d.sort_values("cer", ascending=False).head(10)[["input", "ref_native", "pred_native", "cer"]])

mism = Counter()
for a, b in zip(d.ref_native, d.pred_native):
    for x, y in zip(norm(a).split(), norm(b).split()):
        if x != y:
            mism[(x, y)] += 1
print("Most common word-level errors (reference -> predicted):")
for (x, y), n in mism.most_common(15):
    print(f"  {x} -> {y}   (x{n})")

d = d.copy()
d["sent_chrF"] = [sacrebleu.sentence_chrf(p, [r]).score for p, r in zip(d.pred_en, d.ref_en)]
print("\nLowest translation scores:")
display(d.sort_values("sent_chrF").head(10)[["input", "pred_native", "ref_en", "pred_en", "sent_chrF"]])

# Did the phrasing step change anything (meaning drift)?
p = results["3. + phrasing improve"]
chg = p[p.improved.str.strip() != p.pred_native.str.strip()]
print(f"\nPhrasing step altered {len(chg)}/{len(p)} outputs")
display(chg[["pred_native", "improved", "ref_en", "pred_en"]].head(10))

# Pipeline failure analysis

failed = d[
    d["status"].str.startswith("Error", na=False)
]

print("\nFailed test cases:", len(failed))

if len(failed) > 0:
    display(
        failed[[
            "input",
            "pred_native",
            "status"
        ]]
    )
else:
    print("All test cases completed successfully.")

Worst 10 by CER:


,input,ref_native,pred_native,cer
26,kal se main gym join karunga,कल से मैं जिम जॉइन करूँगा,कल से मैं गिम्नास्टरीक्स जॉइन करूंगा,0.480000
36,yaar tu itna late kyun aaya,यार तू इतना लेट क्यों आया,यार तुम इतना लेट क्यून के आये थे,0.440000
41,main kal Bangalore ja raha hoon,मैं कल बैंगलोर जा रहा हूँ,मैं कल बॉरियाले जा रहा हूँ,0.280000
2,teacher ne homework diya hai,टीचर ने होमवर्क दिया है,शिक्षक ने होमवर्क दिया है,0.260870
7,WiFi kaam nahi kar raha hai,वाईफाई काम नहीं कर रहा है,Wi-Fi काम नहीं कर रहा है,0.240000
18,mujhe thoda time chahiye,मुझे थोड़ा टाइम चाहिए,मुझे थोड़ा समय चाहिए,0.200000
10,meri battery khatam hone wali hai,मेरी बैटरी खत्म होने वाली है,मेरी बैटरी क्षतम होने वाली है,0.142857
38,"kal tak ka time do, main kar dunga","कल तक का टाइम दो, मैं कर दूँगा","कल तक का समय दो, मैं कर दूंगा",0.137931
15,mujhe is project mein aapki help chahiye,मुझे इस प्रोजेक्ट में आपकी हेल्प चाहिए,मुझे इस प्रोजेक्ट में आपकी मदद चाहिए,0.131579
1,mera phone kharab ho gaya hai,मेरा फोन खराब हो गया है,मेरा फोन क्हराब हो गया है,0.130435


Most common word-level errors (reference -> predicted):
  बारिश -> बरिश   (x2)
  टाइम -> समय   (x2)
  कृपया -> कृप्या   (x2)
  खराब -> क्हराब   (x1)
  टीचर -> शिक्षक   (x1)
  वाईफाई -> Wi-Fi   (x1)
  छह -> चौं   (x1)
  खत्म -> क्षतम   (x1)
  प्लेटफॉर्म -> प्लौटफॉर्म   (x1)
  हेल्प -> मदद   (x1)
  पिकनिक -> पिनचिक   (x1)
  बनाएंगे -> बनायenge   (x1)
  लंच -> लूंच   (x1)
  कीजिए -> कीजि�ye   (x1)
  जा -> जाने   (x1)

Lowest translation scores:


,input,pred_native,ref_en,pred_en,sent_chrF
24,mera kaam abhi tak pura nahi hua,मेरा काम अभी तक पूरा नहीं हुआ,My work is not completed yet.,My job isn't done yet.,18.067123
36,yaar tu itna late kyun aaya,यार तुम इतना लेट क्यून के आये थे,"Dude, why did you come so late?","Man, you came so late Cune",34.143086
17,hum sab saath mein lunch karenge,हम सब साथ में लूंच करेंगे,We will all have lunch together.,We're all going to suck together.,34.657845
11,mujhe 2 din ka samay chahiye,मुझे 2 दिन का समय चाहिए,I need two days.,I need 2 days time.,36.982602
1,mera phone kharab ho gaya hai,मेरा फोन क्हराब हो गया है,My phone has stopped working.,My phone has been disconnected.,38.306077
6,yeh kitne ka hai,यह कितने का है,How much is this?,This is how much,40.689648
18,mujhe thoda time chahiye,मुझे थोड़ा समय चाहिए,I need a little time.,I need some time.,40.737708
10,meri battery khatam hone wali hai,मेरी बैटरी क्षतम होने वाली है,My battery is about to run out.,My battery is going to be damaged.,41.683149
21,kripya mujhe jawab dijiye.,कृप्या मुझे जवाब दीजिए।,Please give me an answer.,Please answer me.,44.123539
29,meeting subah 10 baje shuru hogi,मीटिंग सुबह 10 बजे शुरू होगी,The meeting will start at 10 in the morning.,The meeting will begin at 10 am.,45.314712



Phrasing step altered 40/42 outputs


,pred_native,improved,ref_en,pred_en
0,मैं नहीं आ सकता,मैं नहीं आ सकता।,I cannot come.,I can't come.
1,मेरा फोन क्हराब हो गया है,मेरा फोन क्षत्रिय हो गया है,My phone has stopped working.,My phone has become Kshatriya.
2,शिक्षक ने होमवर्क दिया है,शिक्षक ने होमवर्क दिया था,The teacher has given homework.,The teacher had given homework.
3,क्या आप मेरी मदद कर सकते हैं,क्या आप मेरी मदद कर सकते हैं?,Can you help me?,Can you help me?
4,प्लीज़ मुझे एक मिनट दो,प्लीज़ मुझे एक मिनट का समय दें,Please give me a minute.,please give me a minute
5,तुम्हारा नाम क्या है,तुम्हारा नाम क्या है?,What is your name?,What's your name?
6,यह कितने का है,यह कितने रुपये में है,How much is this?,how much is it in rupees
7,Wi-Fi काम नहीं कर रहा है,Wi-Fi काम नहीं कर रहा है।,The WiFi is not working.,Wi-Fi is not working.
8,मुझे कल सुबह चौं बजे उठना है,मुझे कल सुबह चार बजे उठना है।,I have to wake up at six tomorrow morning.,I have to get up at four o'clock tomorrow morn...
9,"मुझे सिर दर्द हो रहा है, डॉक्टर के पास जाना है","मुझे सिर दर्द हो रहा है, डॉक्टर की तरफ जाना है।","I have a headache, I need to go to the doctor.","I have a headache, have to go to the doctor."



Failed test cases: 0
All test cases completed successfully.


## 8. Gradio frontend


In [13]:
import gradio as gr

with gr.Blocks(title="Hinglish Smart Writing Assistant") as demo:
    gr.Markdown("# Hinglish Smart Writing Assistant\nRoman script -> native script -> better phrasing -> English")
    with gr.Row():
        with gr.Column():
            inp = gr.Textbox(lines=5, label="Hinglish input",
                             placeholder="e.g. yaar kal ki meeting reschedule kar do please")
            lang = gr.Dropdown(list(LANGS), value="Hindi", label="Language")
            engine = gr.Radio(available_engines(), value="LLM", label="Transliteration engine")
            improve = gr.Checkbox(value=True, label="Improve phrasing")
            tone = gr.Radio(["neutral", "formal", "casual"], value="neutral", label="Tone")
            to_en = gr.Checkbox(value=True, label="Translate to English")
            keep_en = gr.Checkbox(value=False, label="Keep English words in Latin script (LLM engine)")
            robust = gr.Checkbox(value=True, label="Guardrails + lexicon")
            btn = gr.Button("Convert", variant="primary")
        with gr.Column():
            o1 = gr.Textbox(label="1. Native script", lines=4)
            o2 = gr.Textbox(label="2. Improved phrasing", lines=4)
            o3 = gr.Textbox(label="3. English translation", lines=4)
            status = gr.Textbox(label="Status")
    gr.Examples(
        examples=[["mujhe kal subah 6 baje uthna hai"],
                  ["bhai assignment ka deadline kab hai, mujhe nahi pata"],
                  ["main aaj office nahi aa paunga, tabiyat theek nahi hai"]],
        inputs=[inp],
    )
    btn.click(pipeline, [inp, lang, engine, improve, tone, to_en, keep_en, robust], [o1, o2, o3, status])

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a34c14086b7bb6f3bc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
